# Comparación de motores de decisión — Civil Violence LLM

Explora los resultados generados por `python -m experiments.run_comparison`.
Corré ese script primero (ver TECHNICAL.md) para poblar `results/raw/*.csv`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))

from metrics.report import discover_runs, summarize_engine, FEATURE_MATRIX, PRICING_PER_1K_DECISIONS_USD

RESULTS_DIR = Path("../results/raw")
runs = discover_runs(RESULTS_DIR)
sorted(runs.keys())

## Latencia y tasa de error por motor

In [ ]:
decision_frames = {}
for engine, files in runs.items():
    decision_frames[engine] = pd.concat(
        [pd.read_csv(p) for p in files["decisions"]], ignore_index=True
    )

summary_rows = []
for engine, df in decision_frames.items():
    summary = summarize_engine(df.to_dict(orient="records"))
    summary["engine"] = engine
    summary["cost_usd"] = PRICING_PER_1K_DECISIONS_USD.get(engine, 0.0) * (summary["total_decisions"] / 1000)
    summary_rows.append(summary)

summary_df = pd.DataFrame(summary_rows).set_index("engine").sort_index()
summary_df

In [ ]:
ax = summary_df["mean_latency_ms"].plot(kind="bar", logy=True, color="#648FFF", figsize=(8, 4.5))
ax.set_ylabel("Latencia media por decisión (ms, escala log)")
ax.set_title("Latencia por motor de decisión")
plt.tight_layout()
plt.show()

## Dinámica de la simulación (ciudadanos activos en el tiempo)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for engine, files in runs.items():
    if not files["timeseries"]:
        continue
    ts = pd.read_csv(files["timeseries"][0])
    ax.plot(ts["active_citizens"], label=engine)
ax.set_xlabel("Paso de simulación")
ax.set_ylabel("Ciudadanos activos")
ax.legend()
ax.set_title("Dinámica de ciudadanos activos por motor de decisión")
plt.tight_layout()
plt.show()

## Características cualitativas de cada alternativa

In [ ]:
pd.DataFrame(FEATURE_MATRIX).T